# 02 — PaySim entity and temporal analysis

This notebook evaluates whether PaySim has enough repeated entity history for point-in-time features and proposes the event-ordering contract.

The result is an ADR input, not an automatic decision: `nameOrig` is evaluated as the primary behavioral entity and `nameDest` as recipient context.

In [3]:
from pathlib import Path

from IPython.display import Markdown, display

from pit_fintech.data.paysim import (
    connect_paysim,
    find_paysim_csv,
    resolve_project_root,
    setup_instructions,
)

PROJECT_ROOT = resolve_project_root(Path.cwd())
paysim_csv = find_paysim_csv(PROJECT_ROOT)
DATA_AVAILABLE = paysim_csv is not None

if DATA_AVAILABLE:
    connection = connect_paysim(paysim_csv)
    print({"path": str(paysim_csv), "status": "ready"})
else:
    connection = None
    display(Markdown("```text\n" + setup_instructions(PROJECT_ROOT) + "\n```"))


def show(sql: str):
    if connection is None:
        print("Query skipped: PaySim CSV is unavailable.")
        return None
    table = connection.sql(sql).to_arrow_table()
    display(table)

{'path': 'C:\\workspace\\pit-fintech\\data\\raw\\PS_20174392719_1491204439457_log.csv', 'status': 'ready'}


## Temporal density and deterministic ordering

PaySim only provides an hourly `step`; many events therefore share the same source time. `source_row_number` is a **provisional ingestion tie-break** derived from the original CSV order. Bronze ingestion must persist it once so later runs never recalculate ordering from a rewritten file.

In [4]:
show(
    """
    WITH hourly AS (
        SELECT step, count(*) AS rows, sum(isFraud) AS fraud_rows
        FROM paysim
        GROUP BY step
    )
    SELECT
        count(*) AS observed_steps,
        min(rows) AS min_events_per_step,
        round(avg(rows), 2) AS mean_events_per_step,
        quantile_cont(rows, 0.50) AS p50_events_per_step,
        quantile_cont(rows, 0.95) AS p95_events_per_step,
        max(rows) AS max_events_per_step,
        count_if(fraud_rows > 0) AS steps_with_fraud
    FROM hourly
    """
)

pyarrow.Table
observed_steps: int64
min_events_per_step: int64
mean_events_per_step: double
p50_events_per_step: double
p95_events_per_step: double
max_events_per_step: int64
steps_with_fraud: decimal128(38, 0)
----
observed_steps: [[743]]
min_events_per_step: [[2]]
mean_events_per_step: [[8563.42]]
p50_events_per_step: [[529]]
p95_events_per_step: [[36885.7]]
max_events_per_step: [[51352]]
steps_with_fraud: [[741]]

In [5]:
show(
    """
    WITH signatures AS (
        SELECT
            step, type, amount, nameOrig, nameDest, isFraud, isFlaggedFraud,
            count(*) AS copies
        FROM paysim
        GROUP BY ALL
        HAVING count(*) > 1
    )
    SELECT
        count(*) AS repeated_signatures,
        coalesce(sum(copies - 1), 0) AS extra_rows_with_same_signature
    FROM signatures
    """
)

pyarrow.Table
repeated_signatures: int64
extra_rows_with_same_signature: decimal128(38, 0)
----
repeated_signatures: [[0]]
extra_rows_with_same_signature: [[0]]

## Entity-history viability

A useful PIT workload needs repeated history. The following tables measure singleton rate and history depth instead of assuming that identifier cardinality implies useful temporal behavior.

In [6]:
show(
    """
    WITH entity_counts AS (
        SELECT nameOrig AS entity_id, count(*) AS transaction_count
        FROM paysim
        GROUP BY nameOrig
    )
    SELECT
        count(*) AS origin_entities,
        count_if(transaction_count = 1) AS singleton_entities,
        round(100.0 * count_if(transaction_count = 1) / count(*), 4) AS singleton_percent,
        quantile_cont(transaction_count, 0.50) AS p50_history,
        quantile_cont(transaction_count, 0.90) AS p90_history,
        quantile_cont(transaction_count, 0.99) AS p99_history,
        max(transaction_count) AS max_history
    FROM entity_counts
    """
)

pyarrow.Table
origin_entities: int64
singleton_entities: decimal128(38, 0)
singleton_percent: double
p50_history: double
p90_history: double
p99_history: double
max_history: int64
----
origin_entities: [[6353307]]
singleton_entities: [[6344009]]
singleton_percent: [[99.8537]]
p50_history: [[1]]
p90_history: [[1]]
p99_history: [[1]]
max_history: [[3]]

In [7]:
show(
    """
    WITH entity_counts AS (
        SELECT nameDest AS entity_id, count(*) AS transaction_count
        FROM paysim
        GROUP BY nameDest
    )
    SELECT
        count(*) AS destination_entities,
        count_if(transaction_count = 1) AS singleton_entities,
        round(100.0 * count_if(transaction_count = 1) / count(*), 4) AS singleton_percent,
        quantile_cont(transaction_count, 0.50) AS p50_history,
        quantile_cont(transaction_count, 0.90) AS p90_history,
        quantile_cont(transaction_count, 0.99) AS p99_history,
        max(transaction_count) AS max_history
    FROM entity_counts
    """
)

pyarrow.Table
destination_entities: int64
singleton_entities: decimal128(38, 0)
singleton_percent: double
p50_history: double
p90_history: double
p99_history: double
max_history: int64
----
destination_entities: [[2722362]]
singleton_entities: [[2262704]]
singleton_percent: [[83.1155]]
p50_history: [[1]]
p90_history: [[5]]
p99_history: [[25]]
max_history: [[113]]

In [8]:
show(
    """
    SELECT
        count_if(starts_with(nameOrig, 'C')) AS customer_origin_rows,
        count_if(starts_with(nameDest, 'C')) AS customer_destination_rows,
        count_if(starts_with(nameDest, 'M')) AS merchant_destination_rows,
        count(DISTINCT CASE WHEN starts_with(nameDest, 'M') THEN nameDest END) AS merchants
    FROM paysim
    """
)

pyarrow.Table
customer_origin_rows: decimal128(38, 0)
customer_destination_rows: decimal128(38, 0)
merchant_destination_rows: decimal128(38, 0)
merchants: int64
----
customer_origin_rows: [[6362620]]
customer_destination_rows: [[4211125]]
merchant_destination_rows: [[2151495]]
merchants: [[2150401]]

## Candidate chronological split

This is a viability check, not the final experiment manifest. The split uses time ranges, never random rows.

In [9]:
show(
    """
    WITH bounds AS (
        SELECT max(step) AS max_step FROM paysim
    ), assigned AS (
        SELECT
            CASE
                WHEN step <= floor(0.70 * max_step) THEN 'train'
                WHEN step <= floor(0.85 * max_step) THEN 'validation'
                ELSE 'test'
            END AS split,
            step,
            isFraud
        FROM paysim, bounds
    )
    SELECT
        split,
        min(step) AS min_step,
        max(step) AS max_step,
        count(*) AS rows,
        sum(isFraud) AS fraud_rows,
        round(100.0 * avg(isFraud), 6) AS fraud_percent
    FROM assigned
    GROUP BY split
    ORDER BY min_step
    """
)

pyarrow.Table
split: string
min_step: int64
max_step: int64
rows: int64
fraud_rows: decimal128(38, 0)
fraud_percent: double
----
split: [["train","validation","test"]]
min_step: [[1,521,632]]
max_step: [[520,631,743]]
rows: [[6082007,191147,89466]]
fraud_rows: [[5781,1180,1252]]
fraud_percent: [[0.095051,0.617326,1.399414]]

## ADR candidates to review after execution

1. **Event time:** `step` is an hourly ordinal. Keep it as source time; add a synthetic timestamp anchor only for APIs requiring timestamps.
2. **Tie-break:** persist original `source_row_number` during Bronze ingestion and order by `(step, source_row_number)`.
3. **Knowledge time:** the main PaySim file has no ingestion timestamp. Set `created_timestamp = ordered_event_timestamp` for the main path and use a separate synthetic late-arrival injection for correction tests.
4. **Entity:** use `nameOrig` as the primary behavioral entity if repeated-history results pass the gate. Use `nameDest` for recipient aggregates, not as a composite replacement for the origin.
5. **Cutoff:** score transaction `t` using request fields from `t` plus historical aggregates from events strictly ordered before `t`.